# 00 — Setup Check

Verify the analysis environment: imports, package versions, project paths, and a synthetic-segments smoke test.


In [1]:
%matplotlib inline
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in [str(ROOT), str(ROOT / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)
print(sys.version)


3.13.12 | packaged by Anaconda, Inc. | (main, Feb 24 2026, 16:05:56) [MSC v.1942 64 bit (AMD64)]


In [2]:
import importlib
pkgs = ['pandas', 'numpy', 'geopandas', 'pymc', 'arviz', 'xgboost', 'folium', 'matplotlib']
for p in pkgs:
    try:
        m = importlib.import_module(p)
        print(f'{p:12s} {getattr(m, "__version__", "?")}')
    except ImportError as e:
        print(f'{p:12s} NOT INSTALLED ({e})')


pandas       3.0.2
numpy        2.4.3
geopandas    1.1.3
pymc         NOT INSTALLED (No module named 'pymc')
arviz        NOT INSTALLED (No module named 'arviz')


xgboost      3.2.0
folium       0.20.0
matplotlib   3.10.8


## phillyparking imports

In [3]:
from phillyparking._config import load_config, project_root, data_dir, outputs_dir
from phillyparking.io import opendataphilly, census, lodes, osm_loader, gtfs_loader, ppa_stub
from phillyparking.spatial.segments import build_segments, load_segments, save_segments
from phillyparking.spatial.zones import assign_zones, zone_summary, aggregate_to_zones
from phillyparking.spatial.join import join_acs, join_lodes, join_transit, join_pois
from phillyparking.demand.synthetic_baseline import baseline_occupancy, expand_panel
from phillyparking.demand.occupancy_estimator import transactions_to_occupancy
from phillyparking.demand.ml_occupancy import MLOccupancyModel
from phillyparking.elasticity.priors import central_prior, named_scenario, cross_priors
from phillyparking.elasticity.scenarios import all_scenarios, get_scenario, apply_elasticity
from phillyparking.pricing.seattle_rule import adjust_zone_prices
from phillyparking.pricing.arnott_inci import optimal_price_arnott_inci, cruising_dwl
from phillyparking.pricing.chatman_manville import (
    target_avg_occupancy, target_min_vacancy_peak, compare_targeting_rules
)
from phillyparking.revenue.forecast import annual_revenue, revenue_under_scenarios, revenue_curve
from phillyparking.revenue.allocation import allocate
from phillyparking.welfare.cruising_dwl import annual_cruising_dwl
from phillyparking.welfare.incidence import gini, incidence_summary
from phillyparking.scenarios.runner import run_scenario, run_all_scenarios
from phillyparking.scenarios.compare import comparison_table, chatman_manville_sweep
from phillyparking.viz.charts import (
    revenue_fan_chart, incidence_bar_chart, chatman_manville_results_chart, laffer_curve
)
from phillyparking.viz.maps import segment_map, zone_choropleth, occupancy_heatmap
print('All phillyparking imports OK')


All phillyparking imports OK


## Project paths

In [4]:
print('project_root:', project_root())
print('data_dir   :', data_dir())
print('outputs_dir:', outputs_dir())
try:
    cfg = load_config('zones')
    print('zones config keys:', list(cfg.keys())[:10])
except Exception as e:
    print('load_config(zones) skipped:', e)


project_root: C:\projects\philly-parking-benefit-districts
data_dir   : C:\projects\philly-parking-benefit-districts\data
outputs_dir: C:\projects\philly-parking-benefit-districts\outputs
zones config keys: ['zones', 'rpp']


## Synthetic segments smoke test

In [5]:
from tests.fixtures import make_synthetic_segments
seg = make_synthetic_segments(n=10)
print(seg.shape)
seg.head()


(10, 17)


,segment_id,geometry,length_m,street_name,zone_id,neighborhood,rpp_zone,capacity,has_meter,n_meters,current_rate,median_hh_income,pop_density,vehicles_per_hh,jobs_within_200m,pois_count,transit_trips_per_hr
0,T0000,"LINESTRING (-75.169 39.946, -75.1682 39.946)",92.634142,Synth 0,south_philly,South Philly,None,12,False,0,2.0,33358.557531,29726.906964,0.528352,2596.377821,1,11.988476
1,T0001,"LINESTRING (-75.15967 39.946, -75.15887 39.946)",76.907489,Synth 1,cca,Center City,None,10,True,0,2.0,97062.441469,26593.201440,1.100001,1181.848785,7,39.888397
2,T0002,"LINESTRING (-75.15033 39.946, -75.14953 39.946)",99.233414,Synth 2,ccc,Center City,None,8,True,2,4.0,98844.673057,16779.014111,0.475625,9082.232510,50,12.409675
3,T0003,"LINESTRING (-75.141 39.946, -75.1402 39.946)",79.433414,Synth 3,ccc,Center City,None,10,False,0,4.0,65779.519671,23718.133568,0.718430,8891.450045,56,13.516449
4,T0004,"LINESTRING (-75.169 39.95033, -75.1682 39.95033)",75.664760,Synth 4,fishtown,Fishtown,None,7,True,3,2.0,92318.714469,5192.583056,1.382437,2371.940007,18,35.059369


## Next steps

- Proceed to `01_data_acquisition.ipynb` to pull the real data sources.
- Investigate any package marked NOT INSTALLED.
